# Приложение для поиска фильмов (sakila)

Консольное приложение для поиска фильмов в базе **sakila** (MySQL) с логированием
запросов в **MongoDB**. Каждый раздел ниже можно развернуть кликом по заголовку.

## Импорты

In [1]:
%pip install pymysql pymongo python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv
import pymysql
from pymongo import MongoClient
from pymongo.errors import PyMongoError
from collections import Counter
from datetime import datetime

load_dotenv()  # загружает MYSQL_* и MONGO_URI из .env


## 1. Подключение к MySQL и базе sakila

In [4]:
# Параметры подключения к MySQL.
# Доступы к MySQL читаются из файла .env (он не попадает в git, см. .env.example).
config = {
    "host": os.getenv("MYSQL_HOST"),
    "user": os.getenv("MYSQL_USER"),
    "password": os.getenv("MYSQL_PASSWORD"),
}
db_name = "sakila"


def run_query(sql, params=None):
    """Открывает соединение, выбирает базу sakila, выполняет запрос и
    возвращает все строки. Подключение и выбор базы вынесены сюда, чтобы
    не повторять их в каждой функции поиска (защита от дублирования).
    Параметры подставляются через %s — это защита от SQL-инъекций."""
    with pymysql.connect(**config) as connection:
        with connection.cursor() as cursor:
            cursor.execute(f"USE {db_name}")
            cursor.execute(sql, params or ())
            return cursor.fetchall()

In [5]:
# Проверка подключения.
try:
    with pymysql.connect(**config) as connection:
        with connection.cursor() as cursor:
            cursor.execute(f"USE {db_name}")
    print("MySQL connection successful!")
except pymysql.MySQLError as error:
    print("Не удалось подключиться к MySQL.")
    print("Проверьте параметры подключения (хост, логин, пароль) и доступ к сети.")
    print(f"Подробности: {error}")

MySQL connection successful!


## 2. Поисковые функции (MySQL)

In [6]:
def fetch_films_by_keyword(keyword):
    """Фильмы, название которых содержит keyword.
    Возвращает столбцы (название, жанр, год, рейтинг) через JOIN с category,
    чтобы вывод был таким же, как в остальных сценариях."""
    return run_query(
        """
        SELECT f.title, c.name, f.release_year, f.rating
        FROM film f
        JOIN film_category fc ON f.film_id = fc.film_id
        JOIN category c ON fc.category_id = c.category_id
        WHERE f.title LIKE %s
        ORDER BY f.title
        """,
        ("%" + keyword + "%",),
    )

In [7]:
def fetch_genres():
    rows = run_query("SELECT name FROM category ORDER BY name")
    return [row[0] for row in rows]
def fetch_year_range():
    """Минимальный и максимальный год выпуска в базе: (min_year, max_year)."""
    return run_query("SELECT MIN(release_year), MAX(release_year) FROM film")[0]

In [8]:
def fetch_films_by_genre_year(genre, start_year, end_year):
    """Фильмы заданного жанра за диапазон годов [start_year; end_year].
    Возвращает столбцы (название, жанр, год, рейтинг)."""
    return run_query(
        """
        SELECT f.title, c.name, f.release_year, f.rating
        FROM film f
        JOIN film_category fc ON f.film_id = fc.film_id
        JOIN category c ON fc.category_id = c.category_id
        WHERE c.name = %s AND f.release_year BETWEEN %s AND %s
        ORDER BY f.release_year, f.title
        """,
        (genre, start_year, end_year),
    )

In [9]:
def fetch_ratings():
    """Список доступных возрастных рейтингов из базы (G, PG, PG-13, R, NC-17)."""
    rows = run_query(
        "SELECT DISTINCT rating FROM film WHERE rating IS NOT NULL ORDER BY rating"
    )
    return [row[0] for row in rows]


def fetch_films_by_rating(rating):
    """Фильмы с заданным возрастным рейтингом (дополнительное задание).
    Возвращает столбцы (название, жанр, год, рейтинг)."""
    return run_query(
        """
        SELECT f.title, c.name, f.release_year, f.rating
        FROM film f
        JOIN film_category fc ON f.film_id = fc.film_id
        JOIN category c ON fc.category_id = c.category_id
        WHERE f.rating = %s
        ORDER BY f.title
        """,
        (rating,),
    )

## 3. Функции вывода с пагинацией

In [10]:
PAGE_SIZE = 10  # сколько результатов показываем за один раз

HEADERS = ("No", "Название", "Жанр", "Год", "Рейтинг") # Заголовки колонок таблицы вывода.

def paginate(rows):
    """Выводит строки таблицей порциями по PAGE_SIZE.
    Каждая строка rows — это (название, жанр, год, рейтинг).
    Колонки таблицы: No, Название, Жанр, Год, Рейтинг."""
    if not rows:
        print("По вашему запросу ничего не найдено.")
        return

    total = len(rows)

    # Ширину каждой колонки считаем по самому длинному значению в данных
    # (и по заголовку), чтобы таблица ровно выровнялась — вручную, без
    # сторонних модулей.
    num_width = max(len(HEADERS[0]), len(str(total)))
    title_width = max(len(HEADERS[1]), max(len(str(row[0])) for row in rows))
    genre_width = max(len(HEADERS[2]), max(len(str(row[1])) for row in rows))
    year_width = max(len(HEADERS[3]), max(len(str(row[2])) for row in rows))
    rating_width = max(len(HEADERS[4]), max(len(str(row[3])) for row in rows))

    # Шапка таблицы и линия-разделитель.
    print(
        f"{HEADERS[0]:>{num_width}}  {HEADERS[1]:<{title_width}}  "
        f"{HEADERS[2]:<{genre_width}}  {HEADERS[3]:>{year_width}}  "
        f"{HEADERS[4]:<{rating_width}}"
    )
    print(
        f"{'-' * num_width}  {'-' * title_width}  {'-' * genre_width}  "
        f"{'-' * year_width}  {'-' * rating_width}"
    )

    shown = 0
    while shown < total:
        # Нумеруем строки сквозным номером (продолжается между страницами).
        for number, row in enumerate(rows[shown:shown + PAGE_SIZE], start=shown + 1):
            title, genre, year, rating = row
            print(
                f"{number:>{num_width}}  {str(title):<{title_width}}  "
                f"{str(genre):<{genre_width}}  {str(year):>{year_width}}  "
                f"{str(rating):<{rating_width}}"
            )
        shown += min(PAGE_SIZE, total - shown)
        if shown < total:
            answer = input(
                f"\nПоказано {shown} из {total}. "
                f"Показать ещё? (Enter — да, n — нет): "
            )
            if answer.strip().lower() == "n":
                break

## 4. Подключение к собственной БД MongoDB

In [11]:
# Адрес MongoDB с логином и паролем хранится в .env (MONGO_URI).
mongo_client = MongoClient(os.getenv("MONGO_URI"))
GROUP = "051225"
FULL_NAME = "YerkovskaYuliia"
searches_collection = mongo_client["ich_edit"][f"final_project_{GROUP}_{FULL_NAME}"]

In [12]:
try:
    mongo_client.admin.command("ping")
    print("MongoDB connection successful!")
    print("Коллекция:", searches_collection.name)
except PyMongoError as error:
    print("Не удалось подключиться к MongoDB.")
    print("Проверьте строку подключения (логин, пароль, адрес) и доступ к сети.")
    print(f"Подробности: {error}")

MongoDB connection successful!
Коллекция: final_project_051225_YerkovskaYuliia


## 5. Логирование запросов в MongoDB

In [13]:
def log_search(search_type, params, results_count):
    """Сохраняет один поисковый запрос в MongoDB.

    search_type   — 'keyword', 'genre_year' или 'rating'
    params        — словарь с параметрами запроса
    results_count — сколько фильмов нашлось
    """
    searches_collection.insert_one({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "search_type": search_type,
        "params": params,
        "results_count": results_count,
    })

In [14]:
def _query_label(doc):
    """Превращает документ запроса в читаемую строку 'тип: параметры'."""
    params = doc.get("params", {})
    params_text = ", ".join(f"{key}={value}" for key, value in params.items())
    return f"{doc.get('search_type')}: {params_text}"


def show_statistics():
    """Популярные запросы (по частоте) и последние уникальные запросы."""
    docs = list(searches_collection.find())
    if not docs:
        print("Пока нет сохранённых запросов.")
        return

    # Популярные по частоте — через Counter.
    counter = Counter(_query_label(doc) for doc in docs)
    print("\nТоп-5 популярных запросов (по частоте):")
    for i, (label, count) in enumerate(counter.most_common(5), start=1):
        print(f"{i}. {label} — {count} раз(а)")

    # Последние уникальные запросы: идём от самых свежих и пропускаем повторы,
    # чтобы одинаковые запросы подряд не занимали весь список.
    recent = searches_collection.find().sort("timestamp", -1)
    print("\nПоследние 5 уникальных запросов:")
    seen = set()
    shown = 0
    for doc in recent:
        label = _query_label(doc)
        if label in seen:
            continue
        seen.add(label)
        shown += 1
        print(f"{shown}. [{doc.get('timestamp')}] {label}")
        if shown == 5:
            break

## 6. Функции ввода и валидации:

In [15]:
def ask_year(prompt):
    """Запрашивает год (целое число), повторяя ввод при ошибке."""
    while True:
        value = input(prompt).strip()
        if value.isdigit():
            return int(value)
        print("Нужно ввести год числом. Попробуйте ещё раз.")


def ask_index(prompt, count):
    """Запрашивает номер пункта списка (от 1 до count), повторяя ввод при
    ошибке. Возвращает индекс с нуля (для обращения к списку)."""
    while True:
        value = input(prompt).strip()
        if value.isdigit() and 1 <= int(value) <= count:
            return int(value) - 1
        print(f"Нужно ввести число от 1 до {count}. Попробуйте ещё раз.")

## Собранные сценарии:

In [16]:
def search_by_keyword():
    """Сценарий 1: поиск по ключевому слову (части названия)."""
    keyword = input("Введите ключевое слово (часть названия): ").strip()
    if not keyword:
        print("Пустой запрос.")
        return
    rows = fetch_films_by_keyword(keyword)
    log_search("keyword", {"keyword": keyword}, len(rows))
    paginate(rows)

In [17]:
def search_by_genre_year():
    """Сценарий 2: поиск по жанру и диапазону годов."""
    genres = fetch_genres()
    print("Доступные жанры:")
    for i, genre_name in enumerate(genres, start=1):
        print(f"  {i}. {genre_name}")

    # Жанр выбираем по номеру — так клиент не ошибётся в написании.
    index = ask_index("Введите номер жанра: ", len(genres))
    genre = genres[index]

    min_year, max_year = fetch_year_range()
    print(f"\nГоды выпуска в базе: от {min_year} до {max_year}")

    start_year = ask_year("Введите год начала: ")
    end_year = ask_year("Введите год конца: ")
    if start_year > end_year:  # если перепутали местами — меняем
        start_year, end_year = end_year, start_year

    rows = fetch_films_by_genre_year(genre, start_year, end_year)
    log_search(
        "genre_year",
        {"genre": genre, "start_year": start_year, "end_year": end_year},
        len(rows),
    )
    paginate(rows)

In [18]:
def search_by_rating():
    """Сценарий 3 (доп.): поиск по возрастному рейтингу."""
    ratings = fetch_ratings()
    print("Доступные рейтинги:")
    for rating_name in ratings:
        print(f"  - {rating_name}")

    rating = input("Введите рейтинг: ").strip().upper()
    if rating not in ratings:
        print("Такого рейтинга нет в списке.")
        return

    rows = fetch_films_by_rating(rating)
    log_search("rating", {"rating": rating}, len(rows))
    paginate(rows)

## Основной блок программы с меню

In [19]:
def main_menu():
    """Главный цикл приложения с меню."""
    while True:
        print("\n Поиск фильмов (база sakila) ")
        print("1. Поиск по ключевому слову")
        print("2. Поиск по жанру и диапазону годов")
        print("3. Поиск по рейтингу")
        print("4. Статистика запросов")
        print("0. Выход")

        choice = input("Выберите пункт меню (1, 2, 3, 4 или 0): ").strip()

        try:
            if choice == "1":
                search_by_keyword()
            elif choice == "2":
                search_by_genre_year()
            elif choice == "3":
                search_by_rating()
            elif choice == "4":
                show_statistics()
            elif choice == "0":
                print("Выход из программы.")
                break
            else:
                print("Некорректный выбор. Выберите 1, 2, 3, 4 или 0.")
        except Exception as error:
            # Программа не должна аварийно завершаться из-за ошибки в пункте.
            print(f"Произошла ошибка: {error}")

In [20]:
### main_menu()


 Поиск фильмов (база sakila) 
1. Поиск по ключевому слову
2. Поиск по жанру и диапазону годов
3. Поиск по рейтингу
4. Статистика запросов
0. Выход


Выберите пункт меню (1, 2, 3, 4 или 0):  1
Введите ключевое слово (часть названия):  Fam


No  Название        Жанр        Год  Рейтинг
--  --------------  ---------  ----  -------
 1  CYCLONE FAMILY  Foreign    2008  PG     
 2  DOGMA FAMILY    Animation  1995  G      
 3  FAMILY SWEET    Horror     1993  R      

 Поиск фильмов (база sakila) 
1. Поиск по ключевому слову
2. Поиск по жанру и диапазону годов
3. Поиск по рейтингу
4. Статистика запросов
0. Выход


Выберите пункт меню (1, 2, 3, 4 или 0):  1
Введите ключевое слово (часть названия):  ghdhgd


По вашему запросу ничего не найдено.

 Поиск фильмов (база sakila) 
1. Поиск по ключевому слову
2. Поиск по жанру и диапазону годов
3. Поиск по рейтингу
4. Статистика запросов
0. Выход


...


Топ-5 популярных запросов (по частоте):
1. rating: rating=G — 4 раз(а)
2. keyword: keyword=family — 2 раз(а)
3. keyword: keyword=LOVE — 2 раз(а)
4. keyword: keyword=travel — 2 раз(а)
5. genre_year: genre=Travel, start_year=1990, end_year=2000 — 2 раз(а)

Последние 5 уникальных запросов:
1. [2026-06-18T20:18:18] rating: rating=G
2. [2026-06-18T20:17:04] genre_year: genre=Action, start_year=1990, end_year=2000
3. [2026-06-18T20:16:32] genre_year: genre=Action, start_year=19900, end_year=445646
4. [2026-06-18T20:15:44] keyword: keyword=ghdhgd
5. [2026-06-18T20:15:17] keyword: keyword=Fam

 Поиск фильмов (база sakila) 
1. Поиск по ключевому слову
2. Поиск по жанру и диапазону годов
3. Поиск по рейтингу
4. Статистика запросов
0. Выход


Выберите пункт меню (1, 2, 3, 4 или 0):  4



Топ-5 популярных запросов (по частоте):
1. rating: rating=G — 4 раз(а)
2. keyword: keyword=family — 2 раз(а)
3. keyword: keyword=LOVE — 2 раз(а)
4. keyword: keyword=travel — 2 раз(а)
5. genre_year: genre=Travel, start_year=1990, end_year=2000 — 2 раз(а)

Последние 5 уникальных запросов:
1. [2026-06-18T20:18:18] rating: rating=G
2. [2026-06-18T20:17:04] genre_year: genre=Action, start_year=1990, end_year=2000
3. [2026-06-18T20:16:32] genre_year: genre=Action, start_year=19900, end_year=445646
4. [2026-06-18T20:15:44] keyword: keyword=ghdhgd
5. [2026-06-18T20:15:17] keyword: keyword=Fam

 Поиск фильмов (база sakila) 
1. Поиск по ключевому слову
2. Поиск по жанру и диапазону годов
3. Поиск по рейтингу
4. Статистика запросов
0. Выход


Выберите пункт меню (1, 2, 3, 4 или 0):  0


Выход из программы.


## Отдельные блоки для поиска 

###  Поиск по ключевому слову

In [ ]:
search_by_keyword()

###  Поиск по жанру и диапазону годов

In [ ]:
search_by_genre_year()

###  Поиск по рейтингу (доп. задание)

In [ ]:
search_by_rating()

###  Статистика запросов

In [ ]:
show_statistics()

In [ ]:
mongo_client.close()
print("Соединение с MongoDB закрыто.")